<a href="https://colab.research.google.com/github/Nayab-khalid/FlyRank-AI-Internship/blob/main/work/notebooks/w08_warehouse_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — Warehouse Features and Time-Aware Labels

My Week 5 model was trained on the 30,000-row anonymized starter CSV. Two things
about that bothered me enough to redo it:

1. The features are **contemporaneous** with the label. The starter's 90-day
   aggregates cover the same window the decline label is measured over, so the
   result is an association, not a forecast.
2. The capstone card points at the **full warehouse**, and my Week 4 data
   contract already queries it with DuckDB over `hf://`. There is no good reason
   to model on the small slice.

This notebook rebuilds the feature table from
`fact_content_daily_performance` and defines a label on a **future window**:
features come from days before an anchor date T, the outcome is measured strictly
after T.

**Section 1** surveys what the warehouse actually contains, so the windows are
chosen from the data rather than assumed.

In [ ]:
# ============================================================
# CAPSTONE — SECTION 1
# WHAT IS ACTUALLY IN THE WAREHOUSE
# ============================================================

# REASONING:
# Before choosing a feature window and an outcome window I need to know how many
# months exist, how dense each one is, and whether coverage is stable enough to
# support a future-window label.
#
# This cell reads only aggregates. No row-level data and no private fields.

%pip install -q duckdb huggingface_hub pandas

import os

import duckdb
import pandas as pd

# ------------------------------------------------------------
# TOKEN
# ------------------------------------------------------------
#
# The token is never written into this notebook. In Colab it comes from
# Secrets; locally it comes from the HF_TOKEN environment variable.

HF_TOKEN = None

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    print("Token source: Colab Secrets")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if HF_TOKEN:
        print("Token source: HF_TOKEN environment variable")

if not HF_TOKEN:
    raise RuntimeError(
        "No Hugging Face token found.\n"
        "In Colab: add HF_TOKEN under the key icon in the left sidebar.\n"
        "Locally:  set HF_TOKEN in your environment before starting Jupyter."
    )

# ------------------------------------------------------------
# CONNECT
# ------------------------------------------------------------

con = duckdb.connect()

con.execute(
    "CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN ?)",
    [HF_TOKEN]
)

rel = "hf://datasets/FlyRank/internship-warehouse"

daily = (
    f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet', "
    "hive_partitioning = true)"
)

print("Connected.\n")

# ------------------------------------------------------------
# MONTH-BY-MONTH COVERAGE
# ------------------------------------------------------------

coverage = con.execute(f"""
SELECT
    month,
    COUNT(*)                            AS rows,
    COUNT(DISTINCT client_hash_id)      AS clients,
    COUNT(DISTINCT content_hash_id)     AS pages,
    MIN(report_date)                    AS min_date,
    MAX(report_date)                    AS max_date,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_rows,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_rows
FROM {daily}
GROUP BY month
ORDER BY month
""").df()

print("WAREHOUSE COVERAGE BY MONTH")
print("=" * 78)
print(coverage.to_string(index=False))

print("\nMonths:", len(coverage))
print("Total rows:", f"{coverage['rows'].sum():,}")
print("Date span:", coverage["min_date"].min(), "to", coverage["max_date"].max())


### What I am looking for in the output above

- **How many months**, and whether the row counts are steady or ramping. A
  ramping panel means early months have thinner client coverage.
- **Whether `pages` is stable** across months. If the page population churns
  heavily, a page present in the feature window may be absent from the outcome
  window, and those rows need an explicit decision rather than a silent drop.
- **The GSC vs GA4 split.** Week 4 found GA4 available on only 413,966 of
  9,841,378 March rows, so engagement features will be sparse and must be
  imputed honestly rather than zero-filled.

The feature window and outcome window are chosen in Section 2, once these
numbers are on the page.

## Section 2 — Warehouse layout and the panel warnings

The first attempt at this section failed with a 404 on
`dim_content/**/*.parquet`. `skills/flyrank/flyrank-data/SKILL.md` explains why:
only `fact_content_daily_performance` is partitioned by month. The dimension
tables are single parquet files, so the glob pattern that works for the fact
table does not exist for them.

Rather than guess a second time, this cell lists what is actually in the dataset
repository and prints the dimension schemas from whatever paths it finds.

The same document carries three warnings that change how the feature table has to
be built, and all three are real:

1. **History depth differs wildly per client.** `dim_clients.gsc_data_start`
   records when each client's data begins. A client whose history starts inside
   my feature window would look like a low-traffic page rather than a
   late-joining client, so those clients have to be excluded, not imputed.
2. **GA4 zeros before `ga4_data_start` are fake.** Those rows are zero-filled
   with `ga4_data_available = FALSE`. Summing them treats "not measured" as
   "no engagement", which is the exact mistake the Week 5 data dictionary warned
   about. Every GA4 aggregate must be guarded by the availability flag.
3. **About a third of clients have little usable history.** Filtering is
   expected, and the count that survives should be reported rather than quietly
   dropped.

In [ ]:
# ============================================================
# CAPSTONE — SECTION 2A
# DISCOVER THE LAYOUT, DO NOT GUESS IT
# ============================================================

from huggingface_hub import list_repo_files

REPO_ID = "FlyRank/internship-warehouse"

files = list_repo_files(REPO_ID, repo_type="dataset", token=HF_TOKEN)

print("Files in the dataset repo:", len(files))

# Top-level entries only, so the table layout is visible at a glance.
top_level = sorted({f.split("/")[0] for f in files})

print("\nTop-level entries:")
for entry in top_level:
    n = sum(1 for f in files if f.split("/")[0] == entry)
    kind = "file" if n == 1 and "/" not in [f for f in files
                                            if f.split("/")[0] == entry][0] else "folder"
    print(f"  {entry:<45} {n:>6} file(s)  [{kind}]")


def table_path(name):
    """Return a DuckDB-readable path for a table, whichever layout it uses."""
    partitioned = [f for f in files if f.startswith(name + "/")]
    if partitioned:
        return (f"read_parquet('hf://datasets/{REPO_ID}/{name}/**/*.parquet', "
                "hive_partitioning = true)")
    single = [f for f in files if f == name + ".parquet"]
    if single:
        return f"read_parquet('hf://datasets/{REPO_ID}/{name}.parquet')"
    raise FileNotFoundError(f"Cannot locate table {name} in {REPO_ID}")


daily        = table_path("fact_content_daily_performance")
dim_content  = table_path("dim_content")
dim_clients  = table_path("dim_clients")

print("\nResolved paths:")
print("  daily      :", daily)
print("  dim_content:", dim_content)
print("  dim_clients:", dim_clients)

# ------------------------------------------------------------
# DIMENSION SCHEMAS
# ------------------------------------------------------------

for label, source in (("dim_content", dim_content), ("dim_clients", dim_clients)):
    schema = con.execute(f"DESCRIBE SELECT * FROM {source} LIMIT 1").df()
    print(f"\n{label} columns ({len(schema)}):")
    print(schema[["column_name", "column_type"]].to_string(index=False))

# Remember which columns exist, so Section 2B can use them without guessing.
CLIENT_COLS = set(
    con.execute(f"DESCRIBE SELECT * FROM {dim_clients} LIMIT 1").df()["column_name"]
)
CONTENT_COLS = set(
    con.execute(f"DESCRIBE SELECT * FROM {dim_content} LIMIT 1").df()["column_name"]
)

print("\nHas gsc_data_start:", "gsc_data_start" in CLIENT_COLS)
print("Has ga4_data_start:", "ga4_data_start" in CLIENT_COLS)


### Section 2B — the feature table and the future-window label

| Window | Dates | Used for |
|---|---|---|
| `w3` | 2026-03-03 to 2026-04-01 | feature |
| `w2` | 2026-04-02 to 2026-05-01 | feature |
| `w1` | 2026-05-02 to 2026-05-31 | feature |
| **T** | **2026-06-01** | **prediction point** |
| outcome | 2026-06-01 to 2026-06-30 | label only |

**Why a single global window here, when the guide prefers per-client ones.**
The guide's advice is aimed at defining history windows across a ramping panel.
This task is different: it asks what happens to every page during one specific
future month, so the prediction point is the same date for everyone by
construction. The per-client concern is handled at eligibility instead. A client
whose search history begins inside the feature window is excluded outright,
because its pages would otherwise look like low-traffic pages rather than what
they are.

**The label.** A page is labelled 1 when impressions fall more than 20% from
`w1` to the outcome window. The numerator is measured after T and is unknown at
prediction time; the denominator, `w1`, remains a feature, because knowing a
page's current traffic level is not knowing where it goes next.

**Pages that disappear** are kept and read as zero impressions. Dropping them
would be survivorship bias and would flatter the model.

In [ ]:
# ============================================================
# CAPSTONE — SECTION 2B
# FEATURES BEFORE T, LABEL AFTER T
# ============================================================

T = "2026-06-01"

W1_START, W1_END = "2026-05-02", "2026-05-31"
W2_START, W2_END = "2026-04-02", "2026-05-01"
W3_START, W3_END = "2026-03-03", "2026-04-01"
OUT_START, OUT_END = "2026-06-01", "2026-06-30"

FEATURE_MONTHS = "'2026-03', '2026-04', '2026-05'"
OUTCOME_MONTHS = "'2026-06'"

print("Prediction point T:", T)
print("Feature window:", W3_START, "to", W1_END)
print("Outcome window:", OUT_START, "to", OUT_END)

# ------------------------------------------------------------
# ELIGIBLE CLIENTS
# ------------------------------------------------------------
#
# A client whose GSC history starts inside my feature window has a truncated
# window, so its pages would look artificially quiet. Exclude, do not impute.

if "gsc_data_start" in CLIENT_COLS:
    clients = con.execute(f"""
        SELECT client_hash_id, gsc_data_start
        FROM {dim_clients}
        WHERE gsc_data_start IS NOT NULL
          AND CAST(gsc_data_start AS DATE) <= DATE '{W3_START}'
    """).df()
    eligible_clients = set(clients["client_hash_id"])
    total_clients = con.execute(
        f"SELECT COUNT(*) AS n FROM {dim_clients}").df()["n"].iloc[0]
    print(f"\nClients with GSC history starting on or before {W3_START}: "
          f"{len(eligible_clients)} of {total_clients}")
else:
    eligible_clients = None
    print("\nNo gsc_data_start column; skipping client eligibility filter.")

# ------------------------------------------------------------
# FEATURES — DAYS STRICTLY BEFORE T
# ------------------------------------------------------------
#
# Every GA4 aggregate is guarded by ga4_data_available. Rows before a client's
# ga4_data_start are zero-filled, and counting those zeros would treat
# "not measured" as "no engagement".

feature_sql = f"""
WITH pre AS (
    SELECT *
    FROM {daily}
    WHERE month IN ({FEATURE_MONTHS})
      AND CAST(report_date AS DATE) BETWEEN DATE '{W3_START}' AND DATE '{W1_END}'
)
SELECT
    client_hash_id,
    content_hash_id,

    SUM(gsc_impressions)                                    AS impressions_90d,
    SUM(gsc_clicks)                                         AS clicks_90d,
    SUM(gsc_sum_position)                                   AS sum_position_90d,
    COUNT(DISTINCT CASE WHEN gsc_impressions > 0
                        THEN report_date END)               AS days_with_impressions,

    -- GA4, guarded by the availability flag
    SUM(CASE WHEN ga4_data_available IS TRUE
             THEN ga4_sessions END)                         AS sessions_90d,
    SUM(CASE WHEN ga4_data_available IS TRUE
             THEN ga4_users END)                            AS users_90d,
    SUM(CASE WHEN ga4_data_available IS TRUE
             THEN ga4_engaged_sessions END)                 AS engaged_sessions_90d,
    SUM(CASE WHEN ga4_data_available IS TRUE
             THEN ga4_total_engagement_sec END)             AS engagement_sec_90d,
    SUM(CASE WHEN ga4_data_available IS TRUE
             THEN scroll_events END)                        AS scroll_events_90d,
    SUM(CASE WHEN ga4_data_available IS TRUE
             THEN sessions_ai END)                          AS ai_sessions_90d,
    COUNT(DISTINCT CASE WHEN ga4_data_available IS TRUE AND ga4_sessions > 0
                        THEN report_date END)               AS days_with_sessions,

    -- momentum: three equal 30-day windows, all before T
    SUM(CASE WHEN CAST(report_date AS DATE)
             BETWEEN DATE '{W1_START}' AND DATE '{W1_END}'
             THEN gsc_impressions ELSE 0 END)               AS imp_w1,
    SUM(CASE WHEN CAST(report_date AS DATE)
             BETWEEN DATE '{W2_START}' AND DATE '{W2_END}'
             THEN gsc_impressions ELSE 0 END)               AS imp_w2,
    SUM(CASE WHEN CAST(report_date AS DATE)
             BETWEEN DATE '{W3_START}' AND DATE '{W3_END}'
             THEN gsc_impressions ELSE 0 END)               AS imp_w3,
    SUM(CASE WHEN CAST(report_date AS DATE)
             BETWEEN DATE '{W1_START}' AND DATE '{W1_END}'
             THEN gsc_clicks ELSE 0 END)                    AS clicks_w1,
    SUM(CASE WHEN CAST(report_date AS DATE)
             BETWEEN DATE '{W2_START}' AND DATE '{W2_END}'
             THEN gsc_clicks ELSE 0 END)                    AS clicks_w2,

    MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS had_ga4

FROM pre
GROUP BY client_hash_id, content_hash_id
"""

print("\nBuilding features (streams ~32M daily rows, allow a few minutes)...")
features = con.execute(feature_sql).df()
print("Feature rows:", f"{len(features):,}")

# ------------------------------------------------------------
# LABEL — DAYS ON OR AFTER T ONLY
# ------------------------------------------------------------

outcome = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions_next_30d
FROM {daily}
WHERE month IN ({OUTCOME_MONTHS})
  AND CAST(report_date AS DATE) BETWEEN DATE '{OUT_START}' AND DATE '{OUT_END}'
GROUP BY client_hash_id, content_hash_id
""").df()

print("Outcome rows:", f"{len(outcome):,}")

# ------------------------------------------------------------
# JOIN, FILTER, LABEL
# ------------------------------------------------------------

panel = features.merge(outcome, on=["client_hash_id", "content_hash_id"], how="left")

vanished = int(panel["impressions_next_30d"].isna().sum())
panel["impressions_next_30d"] = panel["impressions_next_30d"].fillna(0)
print(f"\nPages absent from June entirely (read as zero impressions): {vanished:,}")

before_client_filter = len(panel)
if eligible_clients is not None:
    panel = panel[panel["client_hash_id"].isin(eligible_clients)]
    print(f"Rows dropped by client eligibility: "
          f"{before_client_filter - len(panel):,}")

eligible = panel[panel["imp_w1"] > 0].copy()
print(f"Rows dropped for zero impressions in w1: {len(panel) - len(eligible):,}")

eligible["impressions_change_pct"] = (
    (eligible["impressions_next_30d"] - eligible["imp_w1"]) / eligible["imp_w1"] * 100
)
eligible["is_declining_label"] = (eligible["impressions_change_pct"] < -20).astype(int)

print("\n" + "=" * 62)
print("FUTURE-WINDOW PANEL")
print("=" * 62)
print("Pages:", f"{len(eligible):,}")
print("Clients:", eligible["client_hash_id"].nunique())
print("Declining (label = 1):", f"{int(eligible['is_declining_label'].sum()):,}")
print("Base rate:", round(eligible["is_declining_label"].mean(), 4))
print("Went to zero impressions:",
      f"{int((eligible['impressions_next_30d'] == 0).sum()):,}")
print("Median imp_w1:", eligible["imp_w1"].median())
print("GA4 available on:", f"{int(eligible['had_ga4'].sum()):,}",
      "of", f"{len(eligible):,}", "pages")

print("\nLabel distribution:")
display(
    eligible["is_declining_label"].value_counts()
    .rename_axis("is_declining_label").reset_index(name="pages")
)


### What matters in the output above

- **Base rate.** The starter slice was 54.2% declining over a *contemporaneous*
  window. A genuinely forward-looking month may be very different. If it is, the
  two numbers are not comparable and the paper must say so rather than place them
  side by side.
- **How many went to zero.** If that dominates the positive class, the label is
  mostly measuring pages disappearing rather than declining, which is a different
  phenomenon and changes what the recommendation means.
- **How many clients survived eligibility.** The guide predicts about a third of
  clients have little usable history. If far more than that are dropped, the
  window is too early and T should move later.
- **GA4 coverage.** If it is very low, the engagement features carry almost no
  information and belong in the limitations, not the feature table.

### Section 2C — Is the label real?

Section 2B returned a base rate of **0.74**: 162,060 of 219,009 pages losing more
than 20% of their impressions in a single month. That is not a plausible rate for
a real content panel, and an implausible number is a reason to audit, not to
celebrate. Week 5 taught me that the hard way in the opposite direction.

There are three candidate explanations and they need separating before anything
is trained.

**1. June reporting coverage.** The Section 1 coverage table shows GSC-available
rows falling from 4,373,422 in May to 3,878,937 in June while total rows stay
flat. June is also the last month in the warehouse, and final periods are often
still filling. My outcome query sums `gsc_impressions` with no
`gsc_data_available` guard, so a page whose GSC reporting stops in June reads as
zero impressions and is labelled declining. That would make the label partly a
measure of reporting coverage rather than traffic.

**2. Low-volume noise.** Median `imp_w1` is 75 impressions across 30 days, about
2.5 a day. At that volume a swing from 75 to 55 is a 27% "decline" that is really
just noise, and a -20% threshold catches an enormous amount of it.

**3. A genuine seasonal drop.** June may simply be down across this panel. If so
the label is real, but it is a seasonal effect and the paper must say so.

The cell below separates the three. It changes nothing until the answer is known.

In [ ]:
# ============================================================
# CAPSTONE — SECTION 2C
# LABEL SANITY CHECK BEFORE ANY MODELLING
# ============================================================

import numpy as np

# ------------------------------------------------------------
# CHECK 1 — IS JUNE A COMPLETE MONTH?
# ------------------------------------------------------------
#
# If the warehouse was still filling June, daily totals will fall off a cliff
# near the end rather than wobble around a level.

daily_totals = con.execute(f"""
SELECT
    CAST(report_date AS DATE)                         AS d,
    COUNT(*)                                          AS rows,
    SUM(CASE WHEN gsc_data_available IS TRUE
             THEN 1 ELSE 0 END)                       AS gsc_rows,
    SUM(gsc_impressions)                              AS impressions
FROM {daily}
WHERE month IN ('2026-05', '2026-06')
GROUP BY 1
ORDER BY 1
""").df()

print("DAILY TOTALS, MAY AND JUNE 2026")
print("=" * 72)
may = daily_totals[daily_totals["d"].astype(str) < "2026-06-01"]
jun = daily_totals[daily_totals["d"].astype(str) >= "2026-06-01"]

print(f"May  mean daily impressions: {may['impressions'].mean():,.0f}"
      f"   mean gsc_rows: {may['gsc_rows'].mean():,.0f}")
print(f"June mean daily impressions: {jun['impressions'].mean():,.0f}"
      f"   mean gsc_rows: {jun['gsc_rows'].mean():,.0f}")
print(f"\nJune vs May impressions: "
      f"{(jun['impressions'].mean() / may['impressions'].mean() - 1) * 100:+.1f}%")

print("\nLast 10 days of June (looking for a fill-in cliff):")
print(jun.tail(10).to_string(index=False))

# ------------------------------------------------------------
# CHECK 2 — DOES GSC COVERAGE ITSELF DROP FOR OUR PAGES?
# ------------------------------------------------------------
#
# This is the one I suspect. If a page's GSC reporting stops in June, my outcome
# query reads zero impressions and calls it a decline.

coverage_shift = con.execute(f"""
WITH w1 AS (
    SELECT client_hash_id, content_hash_id,
           MAX(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_w1
    FROM {daily}
    WHERE month = '2026-05'
      AND CAST(report_date AS DATE) BETWEEN DATE '2026-05-02' AND DATE '2026-05-31'
    GROUP BY 1, 2
),
out AS (
    SELECT client_hash_id, content_hash_id,
           MAX(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_out
    FROM {daily}
    WHERE month = '2026-06'
    GROUP BY 1, 2
)
SELECT
    COUNT(*)                                              AS pages,
    SUM(CASE WHEN gsc_w1 = 1 THEN 1 ELSE 0 END)           AS gsc_in_may,
    SUM(CASE WHEN gsc_w1 = 1 AND COALESCE(gsc_out, 0) = 0
             THEN 1 ELSE 0 END)                           AS lost_gsc_in_june
FROM w1 LEFT JOIN out USING (client_hash_id, content_hash_id)
""").df()

print("\n" + "=" * 72)
print("GSC COVERAGE SHIFT, MAY -> JUNE")
print("=" * 72)
print(coverage_shift.to_string(index=False))

lost = int(coverage_shift["lost_gsc_in_june"].iloc[0])
in_may = int(coverage_shift["gsc_in_may"].iloc[0])
if in_may:
    print(f"\nPages reporting GSC in May but not in June: "
          f"{lost:,} ({lost / in_may * 100:.1f}%)")

# ------------------------------------------------------------
# CHECK 3 — HOW MUCH OF THE LABEL IS LOW-VOLUME NOISE?
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("BASE RATE BY TRAFFIC LEVEL IN w1")
print("=" * 72)

bands = [(0, 10), (10, 50), (50, 100), (100, 500), (500, 5000), (5000, np.inf)]
rows = []
for lo, hi in bands:
    band = eligible[(eligible["imp_w1"] > lo) & (eligible["imp_w1"] <= hi)]
    if len(band):
        rows.append({
            "imp_w1 band": f"{lo:,}-{hi:,}" if np.isfinite(hi) else f"{lo:,}+",
            "pages": len(band),
            "base_rate": round(band["is_declining_label"].mean(), 4),
            "went_to_zero": int((band["impressions_next_30d"] == 0).sum()),
        })
display(pd.DataFrame(rows))

# ------------------------------------------------------------
# CHECK 4 — THE DISTRIBUTION, NOT JUST THE THRESHOLD
# ------------------------------------------------------------
#
# A -20% threshold hides whether the panel drifted down slightly or collapsed.

pct = eligible["impressions_change_pct"].replace([np.inf, -np.inf], np.nan).dropna()

print("\n" + "=" * 72)
print("IMPRESSION CHANGE, w1 -> JUNE")
print("=" * 72)
for q in (0.10, 0.25, 0.50, 0.75, 0.90):
    print(f"  p{int(q * 100):<3} {pct.quantile(q):>10.1f}%")
print(f"\n  share exactly -100% (went to zero): "
      f"{(pct <= -99.999).mean() * 100:.1f}%")
print(f"  share above 0% (grew):             {(pct > 0).mean() * 100:.1f}%")

# ------------------------------------------------------------
# CHECK 5 — WHAT IF THE OUTCOME IS GUARDED THE SAME WAY?
# ------------------------------------------------------------
#
# Recompute the label counting only pages whose GSC was actually reporting in
# June. If the base rate falls sharply, the original label was measuring
# reporting coverage.

guarded = con.execute(f"""
SELECT client_hash_id, content_hash_id,
       SUM(CASE WHEN gsc_data_available IS TRUE
                THEN gsc_impressions END) AS impressions_next_30d_guarded
FROM {daily}
WHERE month = '2026-06'
  AND CAST(report_date AS DATE) BETWEEN DATE '2026-06-01' AND DATE '2026-06-30'
GROUP BY 1, 2
""").df()

check = eligible.merge(guarded, on=["client_hash_id", "content_hash_id"], how="left")

reported = check[check["impressions_next_30d_guarded"].notna()].copy()
reported["change_guarded"] = (
    (reported["impressions_next_30d_guarded"] - reported["imp_w1"])
    / reported["imp_w1"] * 100
)
reported["label_guarded"] = (reported["change_guarded"] < -20).astype(int)

print("\n" + "=" * 72)
print("BASE RATE, GSC-REPORTING PAGES ONLY")
print("=" * 72)
print("Pages with GSC actually reporting in June:", f"{len(reported):,}",
      "of", f"{len(check):,}")
print("Base rate, unguarded (Section 2B):", round(check["is_declining_label"].mean(), 4))
print("Base rate, GSC-reporting pages only:", round(reported["label_guarded"].mean(), 4))
